# 📈 Embedding Analysis

Visualize and evaluate the Ayurveda knowledge embeddings.

In [ ]:
import json
import numpy as np
from pathlib import Path
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

EMBED_DIR = Path('../data/embeddings')
embeddings = np.load(EMBED_DIR / 'corpus_embeddings.npy')
with open(EMBED_DIR / 'corpus_metadata.json') as f:
    metadata = json.load(f)

print(f'Loaded {embeddings.shape[0]} embeddings of dim {embeddings.shape[1]}')

In [ ]:
# t-SNE visualization
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(embeddings) - 1))
coords = tsne.fit_transform(embeddings)

categories = [m.get('metadata', {}).get('category', 'unknown') for m in metadata]
unique_cats = list(set(categories))
colors = {cat: plt.cm.tab10(i) for i, cat in enumerate(unique_cats)}

plt.figure(figsize=(12, 8))
for cat in unique_cats:
    mask = [c == cat for c in categories]
    pts = coords[mask]
    plt.scatter(pts[:, 0], pts[:, 1], label=cat, alpha=0.7, s=30)

plt.title('AyurVani Knowledge Base — Embedding Space')
plt.legend()
plt.tight_layout()
plt.savefig(EMBED_DIR / 'tsne_visualization.png', dpi=150)
plt.show()

In [ ]:
# Similarity analysis — query examples
import boto3, json

bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')

def embed(text):
    body = json.dumps({'inputText': text, 'dimensions': 1024, 'embeddingTypes': ['float']})
    r = bedrock.invoke_model(body=body, modelId='amazon.nova-embed-multimodal-v1:0', accept='application/json', contentType='application/json')
    return np.array(json.loads(r['body'].read())['embedding'])

queries = [
    'remedy for insomnia and anxiety',
    'herbs for digestive problems',
    'yoga for back pain',
    'Pitta imbalance skin rash',
]

for q in queries:
    qe = embed(q).reshape(1, -1)
    sims = cosine_similarity(qe, embeddings)[0]
    top_idx = np.argsort(sims)[-3:][::-1]
    print(f'\nQuery: "{q}"')
    for idx in top_idx:
        print(f'  [{sims[idx]:.3f}] {metadata[idx]["content"][:100]}...')